# 🚀 Đào tạo mạng SOTA CTR-GCN cho Hệ Thống Báo Động (Transfer Learning)

Notebook này được thiết kế tự động hoàn toàn để bạn có thể tải mã nguồn từ Github (`nhánh dev`), tải weights chuẩn của NTU-RGB+D, biến đổi Data của bạn và tiến hành Huấn luyện Transfer Learning.

In [ ]:
# 1. KẾT NỐI VỚI GOOGLE DRIVE (Nơi bạn up file raw data lên)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. TẢI MÃ NGUỒN TỪ GITHUB (Nhánh dev mới nhất)
!git clone -b dev https://github.com/NBasLongz/realtime-abnormal-behavior-detection.git
%cd realtime-abnormal-behavior-detection
!pip install -r requirements.txt
!pip install onnxruntime

In [ ]:
# 3. CHÉP DỮ LIỆU TỪ DRIVE VÀO PROJECT
# Lưu ý: Bạn cần tạo thư mục 'My Drive/Abnormal_Data/raw' trên Google Drive 
# và chứa các file .mp4 chia theo 3 thư mục: normal, fighting, falling
!cp -r /content/drive/MyDrive/Abnormal_Data/raw/* data/raw/

In [ ]:
# 4. TRÍCH XUẤT KHUNG XƯƠNG (Pose Estimation)
!python scripts/data_processing/build_sequences.py

In [ ]:
# 5. CHUYỂN ĐỔI DATA SANG CHUẨN GCN [N, C, T, V] VÀ AUGMENTATION (x4 Dữ Liệu)
!python scripts/data_processing/export_gcn_dataset.py

In [ ]:
# 6. TẢI WEIGHTS CỦA MÔ HÌNH NTU-RGB+D ĐỂ TRANSFER LEARNING
!mkdir -p weights/classification
!wget -O weights/classification/ctrgcn_ntu_120.pt "https://github.com/Uason-Chen/CTR-GCN/releases/download/v1.0/ctrgcn_ntu_120.pt"
print("Đã tải thành công pretrained weights!")

In [ ]:
# 7. HUẤN LUYỆN MÔ HÌNH (FINE-TUNING) LỚP CUỐI CÙNG
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from core.models.ctr_gcn import CTRGCN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

# Tải data đã xuất
X = np.load('data/processed/gcn/gcn_train_x.npy')
Y = np.load('data/processed/gcn/gcn_train_y.npy')
dataset = TensorDataset(torch.tensor(X).to(device), torch.tensor(Y).to(device))
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Khởi tạo mô hình và tải weights
model = CTRGCN(num_classes=3).to(device)
model.load_pretrained_weights('weights/classification/ctrgcn_ntu_120.pt', device=device)

# Đóng băng các tầng dưới, chỉ train lớp classification layer cuối cùng (Tùy chọn)
# for param in model.parameters(): param.requires_grad = False
# for param in model.fc.parameters(): param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

# Train loop ngắn hạn
model.train()
for epoch in range(1, 21):
    total_loss = 0
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}/20 - Loss: {total_loss/len(loader):.4f}")

# Lưu mô hình PyTorch
torch.save(model.state_dict(), 'weights/classification/ctrgcn_finetuned.pt')
print("✅ Đã huấn luyện xong và lưu mô hình!")

In [ ]:
# 8. XUẤT RA ĐỊNH DẠNG ONNX ĐỂ CHẠY CỰC NHẸ TRÊN CPU CỦA BẠN
dummy_input = torch.randn(1, 3, 30, 17).to(device)
torch.onnx.export(model, dummy_input, 'weights/classification/ctrgcn_best.onnx', 
                  export_params=True, opset_version=12, 
                  input_names=['input'], output_names=['output'])
print("✅ Đã xuất mô hình ra ONNX: weights/classification/ctrgcn_best.onnx")

# Copy về lại Google Drive để tải xuống máy tính cá nhân
!cp weights/classification/ctrgcn_best.onnx /content/drive/MyDrive/Abnormal_Data/